In [ ]:
%%writefile app.py
import streamlit as st
import seaborn as sns

st.set_page_config(page_title="타이타닉 생존 대시보드", layout="wide")

@st.cache_data
def load_data():
    return sns.load_dataset("titanic")

df = load_data()

def filter_data(df, sexes, classes, age_range, ports):
    f = df[df["sex"].isin(sexes)
           & df["pclass"].isin(classes)
           & df["embarked"].isin(ports)]          # [추가] ① 항구 필터
    if age_range != (0, 80):
        f = f[f["age"].between(*age_range)]
    return f

def kpi(f):
    n = len(f)
    rate = round(f["survived"].mean() * 100, 1) if n else 0.0
    age = round(f["age"].mean(), 1) if n else 0.0
    return n, rate, age

# ---------------- 사이드바 ----------------
st.sidebar.header("필터")
sexes = st.sidebar.multiselect("성별", ["male", "female"], ["male", "female"])
classes = st.sidebar.multiselect("객실 등급", [1, 2, 3], [1, 2, 3])
ages = st.sidebar.slider("나이 범위", 0, 80, (0, 80))
ports = st.sidebar.multiselect("승선 항구",                # [추가] ② 항구 multiselect
                               ["S", "C", "Q"], ["S", "C", "Q"])

f = filter_data(df, sexes, classes, ages, ports)          # [추가] ports 전달
n, rate, age = kpi(f)

# ---------------- 본문 ----------------
st.title("🚢 타이타닉 생존 분석 대시보드")
c1, c2, c3 = st.columns(3)
c1.metric("승객 수", f"{n}명")
c2.metric("생존율", f"{rate}%",
          f"{rate - 38.4:+.1f}%p")                        # [추가] ③ 전체 대비 delta
c3.metric("평균 나이", f"{age}세")

st.dataframe(f, use_container_width=True)

Overwriting app.py


In [7]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!streamlit run app.py &>/content/logs.txt & ./cloudflared tunnel --url http://localhost:8501

2026-11-24T01:27:13Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-24T01:27:13Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-24T01:27:16Z INF +--------------------------------------------------------------------------------------------+
2026-11-24T01:27:16Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-24T01:27:16Z INF |  https://simon-returning-eclipse-replied.trycloudflare